In [3]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool
from langchain.agents import create_agent
load_dotenv()

C:\Users\eyosi\AI_TUTORIAL\tutorila-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
llm = ChatAnthropic(
    model="claude-sonnet-4-5"
)

PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_products(name: str) -> str:
    """Look up a product by name and return its price, rating, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

agent = create_agent(
    llm,
    tools=[get_products],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [6]:
ask("what is the price of wireless headphones.")

The wireless headphones are priced at **$79.99**. 

They feature:
- Over-ear Bluetooth design
- 30-hour battery life
- Active noise cancellation
- Customer rating: 4.6/5

Would you like more information about this product or help with anything else?


In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [8]:
agent2 = create_agent(
    llm,
    tools=[get_products, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [9]:
ask2("how do people like smart watch")

Based on the reviews, the **smart watch** has:
- **340 reviews** from customers
- **3.9 out of 5 rating**

This indicates that people generally have a **positive opinion** of smart watches, with the rating being just under 4 stars. The large number of reviews (340) suggests it's a popular product with good customer engagement.

The 3.9 rating suggests that while most customers are satisfied, there's room for improvement. This is fairly typical for tech products where experiences can vary based on individual needs, compatibility with phones, battery life expectations, and specific features people are looking for.

Would you like to know more details about the smart watch, such as its price, features, or description?


In [10]:
from langgraph.checkpoint.memory import InMemorySaver

In [17]:
agent2 = create_agent(
    llm,
    tools=[get_products, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]
        }, config
    )
    print(result["messages"][-1].content)

In [18]:
ask2("what is the price of wireless headphones.")

The **wireless headphones** are priced at **$79.99**.

They have a rating of 4.6 stars and feature:
- Over-ear Bluetooth design
- 30-hour battery life
- Active noise cancellation

Would you like to know more about this product or see customer reviews?
